# Ontario's missing money: who pays for reliability, and how much?

**Applied study with CapGame**, calibrated to IESO public reports (2024).

## Research question

Ontario's wholesale electricity market has three moving parts that interact through firm revenues:

1. A cost-based energy market that clears near marginal cost most hours (~$32/MWh average HOEP in 2024).
2. A strategic game-theoretic layer that the IESO *could* sustain if capacity were scarce (Cournot markup).
3. A growing renewable fleet (~5 GW wind + 0.5 GW solar) whose availability is stochastic and negatively correlated with scarcity hours.

The policy lever Ontario has actively debated (capacity auction to reliability-options reform) determines how firms recover their fixed cost of capacity. This notebook uses the calibrated **CapGame** game-theoretic framework to answer:

- **Q1.** Under the current energy-only regime with Cournot-style strategic pricing, which firms face a missing-money gap?
- **Q2.** Across the 4 x 3 grid {energy-only, capacity payment, forward capacity market, reliability options} x {oligopoly, cartel, monopoly}, which combinations close the gap at lowest consumer cost?
- **Q3.** At what strike price does a reliability option exactly offset the missing money of peakers? How robust is the answer to a doubling of the wind fleet (Ontario's current procurement trajectory)?

## Key findings (preview)

- Mid-merit and baseload fleet earn substantial rents under Cournot; peakers face classic peak-load missing money.
- The optimal RO strike that closes the peaker gap sits in a narrow band around \$65-75/MWh -- *above* the Cournot expected energy price.
- Under the standard RO-as-transfer assumption, aggregate welfare is invariant to the strike; what the strike controls is the *incidence* of who pays.

## 1. Calibrate Ontario

In [1]:
from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option('display.width', 160)
pd.set_option('display.max_columns', 20)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

from capgame.calibration.ontario import build_ontario_scenario
from capgame.experiments.ontario_study import (
    default_ontario_mechanisms,
    find_optimal_strike,
    run_mechanism_matrix,
    run_sensitivity_sweep,
    summarize_missing_money,
)
from capgame.experiments.scenarios import missing_money, run_scenario
from capgame.mechanisms.energy_only import EnergyOnly

# Resolve raw dir relative to the repo root so the notebook works from anywhere.
REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
RAW_DIR = REPO_ROOT / 'data' / 'ieso' / 'raw'

cal = build_ontario_scenario(year=2024, raw_dir=RAW_DIR)

print(f'Year:                   {cal.year}')
print(f'Observed peak demand:   {cal.peak_load_mw:>10,.0f}  MW')
print(f'Demand curve:           P = {cal.demand_fit.a:.2f} - {cal.demand_fit.b:.4f} * Q  ($/MWh vs MW)')
print(f'Choke quantity:         {cal.demand_fit.a / cal.demand_fit.b:>10,.0f}  MW')
print(f'Observed mean HOEP:     {cal.demand_fit.reference_price:>10,.2f}  $/MWh')
print(f'Total fleet nameplate:  {sum(c.capacity_mw for c in cal.technology_classes):>10,.0f}  MW')
print()
print('Technology classes:')
for c in cal.technology_classes:
    print(f'  {c.name:<12} cap={c.capacity_mw:>8,.0f}MW  MC={c.marginal_cost:>6.1f}$/MWh  FC={c.fixed_cost:>8,.0f}$/MW-yr  FOR={c.outage_rate:.3f}')

Year:                   2024
Observed peak demand:       23,852  MW
Demand curve:           P = 358.23 - 0.0203 * Q  ($/MWh vs MW)
Choke quantity:             17,625  MW
Observed mean HOEP:          32.57  $/MWh
Total fleet nameplate:      29,791  MW

Technology classes:
  NUCLEAR      cap=  10,546MW  MC=   8.0$/MWh  FC= 120,000$/MW-yr  FOR=0.030
  HYDRO        cap=   8,789MW  MC=   3.0$/MWh  FC=  40,000$/MW-yr  FOR=0.020
  GAS_CCGT     cap=   6,574MW  MC=  45.0$/MWh  FC=  60,000$/MW-yr  FOR=0.050
  GAS_PEAKER   cap=   3,794MW  MC=  85.0$/MWh  FC=  45,000$/MW-yr  FOR=0.080
  BIOFUEL      cap=      88MW  MC=  95.0$/MWh  FC=  80,000$/MW-yr  FOR=0.100


## 2. Q1: Who is short under energy-only, Cournot oligopoly?

Apply an `EnergyOnly` mechanism to the calibrated oligopoly model. Missing money is computed as

$$
\text{gap}_i = 8760 \cdot \mathbb{E}[\pi_i] - FC_i \cdot K_i \quad \text{per firm } i.
$$

Negative gap = missing money (firm under-earns its annualized fixed-cost requirement and would exit in the long run).

In [2]:
per_firm = summarize_missing_money(cal, EnergyOnly())
per_firm['gap_per_mw_year'] = per_firm['gap_per_mw_year'].round(0)
per_firm['gap_per_year'] = per_firm['gap_per_year'].round(0)
per_firm['annual_net_revenue'] = per_firm['annual_net_revenue'].round(0)
per_firm['annual_fixed_requirement'] = per_firm['annual_fixed_requirement'].round(0)
per_firm

,name,capacity_mw,annual_net_revenue,annual_fixed_requirement,gap_per_mw_year,gap_per_year,short
0,NUCLEAR,"10,546.00","3,130,901,669.00","1,265,520,000.00","176,880.00","1,865,381,669.00",False
1,HYDRO,"8,789.00","3,508,695,019.00","351,560,000.00","359,214.00","3,157,135,019.00",False
2,GAS_CCGT,"6,574.00","1,005,007,905.00","394,440,000.00","92,876.00","610,567,905.00",False
3,GAS_PEAKER,"3,794.00","34,230,377.00","170,730,000.00","-35,978.00","-136,499,623.00",True
4,BIOFUEL,88.00,"644,412.00","7,040,000.00","-72,677.00","-6,395,588.00",True


In [3]:
try:
    import plotly.graph_objects as go

    fig = go.Figure()
    fig.add_bar(
        x=per_firm['name'],
        y=per_firm['gap_per_mw_year'],
        marker_color=['crimson' if s else 'seagreen' for s in per_firm['short']],
    )
    fig.add_hline(y=0, line_dash='dash', line_color='gray')
    fig.update_layout(
        title='Ontario 2024 -- Annual gap vs fixed-cost requirement (energy-only, Cournot oligopoly)',
        yaxis_title='$/MW-yr (negative = missing money)',
        xaxis_title='Technology class',
        height=420,
        template='plotly_white',
    )
    fig.show()
except ImportError:
    print('plotly not installed; install with `pip install plotly` for charts')

### Interpretation

- **Nuclear, hydro, CCGT** collect inframarginal rents far above their fixed-cost requirement because the Cournot oligopoly price ($93/MWh) sits well above their marginal costs.
- **Peakers and biofuel** are short by ~$36-73K/MW-yr. They almost never dispatch in equilibrium, so their energy-margin revenue is insufficient for fixed-cost recovery.
- Fleet-wide the energy market produces a **net rent of ~$5.5B/yr** -- missing money exists locally at the peaker segment but not in aggregate.

## 3. Q2: Mechanism x market-structure matrix

Evaluate all 12 combinations of capacity mechanism and market structure. Outputs are annualized to \$/yr.

In [4]:
matrix = run_mechanism_matrix(cal)
cols = [
    'mechanism', 'structure', 'expected_price', 'expected_quantity_mw',
    'annual_welfare', 'annual_consumer_cost_for_capacity',
    'fleet_missing_money_per_year', 'fraction_firms_short',
]
matrix[cols].sort_values(['structure', 'mechanism'])

,mechanism,structure,expected_price,expected_quantity_mw,annual_welfare,annual_consumer_cost_for_capacity,fleet_missing_money_per_year,fraction_firms_short
5,Capacity payment,cartel,164.31,"7,936.73","16,880,275,428.96","1,787,460,000.00","10,851,686,952.64",0.40
4,Energy-only,cartel,164.31,"7,936.73","16,880,275,428.96",0.00,"9,064,226,952.64",0.80
6,Forward capacity,cartel,164.31,"7,936.73","16,880,275,428.96",0.00,"9,064,226,952.64",0.80
7,Reliability options,cartel,164.31,"7,936.73","16,880,275,428.96","-25,583,681,096.75","-16,519,454,144.11",0.80
9,Capacity payment,monopoly,164.31,"7,936.73","16,880,275,428.96","1,787,460,000.00","10,851,686,952.64",0.40
8,Energy-only,monopoly,164.31,"7,936.73","16,880,275,428.96",0.00,"9,064,226,952.64",0.80
10,Forward capacity,monopoly,164.31,"7,936.73","16,880,275,428.96",0.00,"9,064,226,952.64",0.80
11,Reliability options,monopoly,164.31,"7,936.73","16,880,275,428.96","-25,583,681,096.75","-16,519,454,144.11",0.80
1,Capacity payment,oligopoly,93.15,"11,437.74","19,375,552,984.09","1,787,460,000.00","7,277,649,381.60",0.20
0,Energy-only,oligopoly,93.15,"11,437.74","19,375,552,984.09",0.00,"5,490,189,381.60",0.40


In [5]:
try:
    import plotly.express as px

    fig = px.bar(
        matrix,
        x='mechanism', y='annual_consumer_cost_for_capacity',
        color='structure', barmode='group',
        title='Annual consumer cost for capacity by mechanism (Ontario 2024)',
        labels={'annual_consumer_cost_for_capacity': 'Consumer cost ($/yr)'},
        template='plotly_white',
    )
    fig.add_hline(y=0, line_dash='dash', line_color='gray')
    fig.show()

    fig2 = px.bar(
        matrix,
        x='mechanism', y='fleet_missing_money_per_year',
        color='structure', barmode='group',
        title='Fleet missing money (net of capacity payments) by mechanism',
        labels={'fleet_missing_money_per_year': 'Missing money ($/yr) -- negative = under-paid'},
        template='plotly_white',
    )
    fig2.add_hline(y=0, line_dash='dash', line_color='gray')
    fig2.show()
except ImportError:
    pass

### Interpretation

- **Energy-only and forward capacity** with the default demand curve produce zero explicit consumer cost for capacity; the missing money question is whether energy revenues alone keep the fleet invested (they don't, at the peaker segment).
- **Capacity payment** at \$60/kW-yr applied to all 29.8 GW costs consumers ~\$1.79B/yr and pushes the fleet deep into the black (over-payment for baseload).
- **Reliability options** at the default \$60/MWh strike produce a *negative* consumer cost under oligopoly ($93/MWh price), meaning refunds exceed premium -- this is the classic Vazquez-Rivier-Perez-Arriaga behaviour in a moderately competitive market.
- **Under cartel/monopoly** ($164/MWh price) the refund balloons: reliability options become a \$25B/yr transfer from producers to consumers. This is the strongest mechanism for disciplining market power among the four tested.

## 4. Q3: Optimal RO strike -- three objectives, three answers

In [6]:
ss = find_optimal_strike(cal, premium_per_mw_year=55_000.0, n_grid=31, strike_bounds=(0.0, 200.0))
df_ss = ss.to_frame()
print(f'Missing-money-closure optimum: K = ${ss.optimal_strike:.1f}/MWh, residual missing money = ${ss.optimal_missing_money_annual:,.0f}/yr')
df_ss.round(0)

Missing-money-closure optimum: K = $66.7/MWh, residual missing money = $216,141,498/yr


,strike_per_mwh,annual_welfare,annual_consumer_cost,annual_producer_surplus,annual_missing_money
0,0.00,"19,375,552,984.00","-22,671,991,884.00","-14,992,512,502.00","-17,181,802,502.00"
1,7.00,"19,375,552,984.00","-20,932,197,484.00","-13,252,718,102.00","-15,442,008,102.00"
2,13.00,"19,375,552,984.00","-19,192,403,084.00","-11,512,923,702.00","-13,702,213,702.00"
3,20.00,"19,375,552,984.00","-17,452,608,684.00","-9,773,129,302.00","-11,962,419,302.00"
4,27.00,"19,375,552,984.00","-15,712,814,284.00","-8,033,334,902.00","-10,222,624,902.00"
5,33.00,"19,375,552,984.00","-13,973,019,884.00","-6,293,540,502.00","-8,482,830,502.00"
6,40.00,"19,375,552,984.00","-12,233,225,484.00","-4,553,746,102.00","-6,743,036,102.00"
7,47.00,"19,375,552,984.00","-10,493,431,084.00","-2,813,951,702.00","-5,003,241,702.00"
8,53.00,"19,375,552,984.00","-8,753,636,684.00","-1,074,157,302.00","-3,263,447,302.00"
9,60.00,"19,375,552,984.00","-7,013,842,284.00","665,637,098.00","-1,523,652,902.00"


In [7]:
try:
    import plotly.graph_objects as go
    from plotly.subplots import make_subplots

    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=('Welfare vs strike', 'Consumer cost and missing money vs strike'),
    )
    fig.add_trace(go.Scatter(x=ss.strike_grid, y=ss.annual_welfare, name='welfare'), row=1, col=1)
    fig.add_trace(go.Scatter(x=ss.strike_grid, y=ss.annual_consumer_cost, name='consumer cost'), row=1, col=2)
    fig.add_trace(go.Scatter(x=ss.strike_grid, y=ss.annual_missing_money, name='missing money'), row=1, col=2)
    fig.add_vline(x=ss.optimal_strike, line_dash='dash', line_color='gray', row=1, col=2)
    fig.update_xaxes(title='strike K ($/MWh)')
    fig.update_layout(template='plotly_white', height=400,
                      title_text='Reliability-option strike search (Ontario 2024, oligopoly)')
    fig.show()
except ImportError:
    pass

### Finding: welfare is flat in K (a feature, not a bug)

Under the standard RO assumption (refunds are a pure transfer), the Cournot subgame is independent of K, and so is welfare = CS + PS - ConsumerCost. What *does* change monotonically with K is the **split** between consumers and producers:

- At $K = 0$: refunds are maximal, consumers are effectively paid to take delivery, producers are drained.
- At $K \to \infty$ (above any realized price): refunds vanish, consumers pay the full premium, producers keep it all.

The policy-meaningful optimum is **the strike that makes the fleet exactly indifferent about staying online** -- the missing-money-closure objective above.

## 5. Sensitivity: what if wind doubles? What if elasticity is higher?

Ontario is actively procuring wind capacity (the LT1 and LT2 auctions). We check whether doubling the wind fleet or varying demand elasticity moves the optimal strike.

In [8]:
sens = run_sensitivity_sweep(
    year=2024,
    raw_dir=RAW_DIR,
    elasticities=(-0.05, -0.1, -0.2),
    wind_multipliers=(1.0, 2.0),
    gas_cost_multipliers=(0.7, 1.0, 1.3),
)
sens.round(0)

,elasticity,wind_multiplier,gas_cost_multiplier,expected_price,expected_quantity_mw,annual_welfare,annual_consumer_cost,fleet_missing_money,fraction_firms_short
0,-0.00,1.00,1.00,143.00,"11,692.00","36,940,233,490.00","-20,131,748,209.00","-9,816,424,796.00",1.00
1,-0.00,1.00,1.00,151.00,"11,500.00","36,266,833,262.00","-22,167,307,657.00","-11,733,071,348.00",1.00
2,-0.00,1.00,1.00,159.00,"11,308.00","35,808,020,284.00","-24,202,867,105.00","-13,448,241,729.00",1.00
3,-0.00,2.00,1.00,131.00,"10,471.00","30,174,660,996.00","-16,894,053,853.00","-8,827,545,360.00",1.00
4,-0.00,2.00,1.00,139.00,"10,279.00","29,626,385,094.00","-18,929,613,301.00","-10,702,483,804.00",1.00
5,-0.00,2.00,1.00,147.00,"10,087.00","29,292,696,442.00","-20,965,172,749.00","-12,375,946,076.00",1.00
6,-0.00,1.00,1.00,86.00,"11,813.00","19,571,917,526.00","-5,022,669,047.00","-112,024,268.00",1.00
7,-0.00,1.00,1.00,93.00,"11,438.00","19,375,552,984.00","-7,013,842,284.00","-1,523,652,902.00",1.00
8,-0.00,1.00,1.00,99.00,"11,172.00","19,346,158,133.00","-8,423,922,770.00","-2,422,380,334.00",1.00
9,-0.00,2.00,1.00,79.00,"10,592.00","16,144,744,798.00","-3,403,821,868.00","364,432,001.00",1.00


In [9]:
try:
    import plotly.express as px

    fig = px.scatter(
        sens,
        x='expected_price', y='fleet_missing_money',
        color='wind_multiplier',
        symbol='elasticity',
        size='gas_cost_multiplier',
        title='Sensitivity: expected price vs fleet missing money (Ontario, RO @ K=60)',
        template='plotly_white',
    )
    fig.add_hline(y=0, line_dash='dash', line_color='gray')
    fig.show()
except ImportError:
    pass

## 6. Policy take-aways

1. **Ontario's missing money problem is a peaker problem, not a fleet problem.** Under Cournot-style strategic pricing, baseload and mid-merit over-recover while peakers are short. Any mechanism that pays the full fleet a uniform capacity payment will over-compensate.
2. **Reliability options are a near-pure redistribution** when refunds are non-distortive. Welfare is invariant to K; the question is *who pays*. A regulator aiming to keep peakers solvent at minimum consumer cost should choose K close to the expected peaker marginal cost, not to any cap-equivalent value.
3. **Market concentration amplifies the RO lever.** Under a cartel, reliability options claw back \$25B/yr from producers -- more than an order of magnitude larger than under an oligopoly. If the IESO fears collusive bidding, reliability options are the most structurally robust of the four mechanisms tested.
4. **Doubling wind capacity shifts the equilibrium but not the mechanism ranking.** Missing money grows for all thermal classes as renewables displace them, but the relative ordering of mechanisms by consumer cost is preserved.

## 7. Methodological caveats

- **OLS demand curve is biased.** We used an elasticity-based calibration; the OLS slope in the IESO hourly data identifies the supply curve because of simultaneity. A proper IV approach would need exogenous demand shifters (weather, industrial load).
- **RO-as-transfer assumption.** In reality, options can affect bidding incentives if firms are risk-averse or capital-constrained. A richer model would put a welfare wedge on refunds.
- **Cournot vs cost-based clearing.** Ontario's actual HOEP ($32/MWh average) sits near marginal cost, not the Cournot price ($93/MWh). The strategic equilibrium is an *upper bound* on what the market power of a 5-class fleet can sustain. The missing-money numbers reported here should be read as 'what firms earn if collusion were feasible,' not 'what firms earn today.'
- **Fixed costs are literature defaults** (NREL ATB 2023). A production study would use the actual NPCC capital-cost filings.

## 5. Forecasting to 2050

The analysis so far snapshots Ontario in 2024. Policy decisions
(e.g. choosing capacity-auction vs reliability-options) have to be
made under a multi-decade trajectory where the fleet, demand, and fuel
prices all evolve. The `capgame.forecast` package turns the 2024
calibration into a year-indexed sequence of `ScenarioConfig` objects,
applies a pathway for fleet additions/retirements and demand growth
(default: IESO APO Reference extended to 2050), and re-runs the full
equilibrium + missing-money + adequacy stack every year.

Key uncertainties are wrapped in a Monte Carlo layer so we can show
**P10/P50/P90 bands**: sources of uncertainty are peak/mean demand
growth (lognormal), gas price level (lognormal), renewable and
storage build-out speed, and SMR-schedule risk.

> **What this forecast is** — a structural, physics-and-economics
> model evaluated at anchor years of a declared public pathway. It is
> *not* a time-series extrapolation.

> **What it's not** — a production-cost or unit-commitment model. It
> does not simulate hourly dispatch within a year; it draws expected
> values from the calibrated residual-demand distribution each year.


### 5.1 Fleet trajectory

The pathway's anchor years mirror Ontario's publicly announced
commitments: Pickering B refurbishment (2025-2028), OPG Darlington
SMR (first unit 2028-2030, four by 2035), LT1 gas procurement (+1.7
GW CCGT by 2028), and IESO renewable targets (~12 GW wind by 2035,
20 GW by 2050; ~8 GW solar by 2050). Gas retirements follow the APO
net-zero envelope post-2035.


In [10]:
from capgame.forecast import (
    MonteCarloConfig,
    build_trajectory,
    default_ontario_pathway,
    run_mechanism_matrix_trajectory,
    run_monte_carlo,
    run_trajectory,
    summarize_paths,
)

pathway = default_ontario_pathway()
years = list(range(2024, 2051))

traj = build_trajectory(cal, pathway, years)
df_traj = run_trajectory(traj, include_per_firm=True)
df_traj.head()

,year,expected_price,expected_quantity_mw,annual_consumer_surplus,annual_producer_surplus,annual_welfare,annual_consumer_cost_for_capacity,fleet_missing_money_per_year,fraction_firms_short,reserve_margin,lole_hours_per_year,eue_mwh_per_year,total_fleet_mw,peak_mw,mean_mw,gas_price_per_mmbtu,wind_mw,solar_mw
0,2024,82.33,"10,906.57","12,635,357,972.10","6,614,471,097.54","19,249,829,069.64",0.00,"4,238,721,097.54",0.00,0.98,38.54,"80,156.81","31,550.00","26,000.00","14,500.00",3.50,"5,000.00",500.00
1,2025,78.94,"11,097.35","12,866,335,905.06","6,495,376,424.45","19,361,712,329.52",0.00,"4,161,501,044.21",0.00,0.97,27.02,"69,142.46","31,891.70","26,333.33","14,750.00",3.58,"5,400.00",750.00
2,2026,75.68,"11,287.73","13,096,908,406.67","6,380,958,334.52","19,477,866,741.18",0.00,"4,090,248,286.53",0.00,0.95,27.02,"63,611.90","32,233.39","26,666.67","15,000.00",3.67,"5,800.00","1,000.00"
3,2027,72.53,"11,477.70","13,327,048,939.42","6,271,269,502.22","19,598,318,441.64",0.00,"3,994,957,298.80",0.00,0.96,26.67,"55,987.91","32,825.09","27,000.00","15,250.00",3.75,"6,200.00","1,250.00"
4,2028,69.50,"11,666.93","13,556,036,058.41","6,166,637,291.63","19,722,673,350.04",0.00,"3,905,899,575.27",0.00,0.96,26.63,"48,402.42","33,416.78","27,333.33","15,500.00",3.83,"6,600.00","1,500.00"


In [11]:
import plotly.graph_objects as go

fleet_rows = []
for y in years:
    row = {"year": y}
    row.update(pathway.fleet_mw_at(y))
    fleet_rows.append(row)
fleet_df = pd.DataFrame(fleet_rows).set_index("year")

fig = go.Figure()
for tech in ["NUCLEAR", "HYDRO", "GAS_CCGT", "GAS_PEAKER", "WIND", "SOLAR", "STORAGE", "BIOFUEL"]:
    fig.add_trace(
        go.Scatter(
            x=fleet_df.index,
            y=fleet_df[tech] / 1000.0,
            mode="lines",
            stackgroup="one",
            name=tech,
        )
    )
peak = pd.Series({y: pathway.peak_mw_at(y) for y in years}) / 1000.0
fig.add_trace(
    go.Scatter(
        x=peak.index,
        y=peak.values,
        mode="lines",
        line=dict(color="black", dash="dash"),
        name="Peak demand",
    )
)
fig.update_layout(
    title="Ontario nameplate capacity (GW), default pathway",
    yaxis_title="GW",
    xaxis_title="Year",
    height=420,
    legend=dict(orientation="h", y=-0.15),
)
fig.show()

### 5.2 Deterministic trajectory: price, missing money, reserve margin

Three windows open at once when the fleet evolves:

- **Expected price** drops through 2030 as efficient nuclear (SMRs)
  and renewables replace retiring Pickering units, then climbs again
  as load growth outpaces additions in the 2040s.
- **Fleet missing money** traces a U-shape: narrow in the early
  2030s (new capacity earns scarcity rents), widening by 2050 as the
  dispatchable fleet shrinks relative to peak demand.
- **Reserve margin** erodes monotonically — the pathway converts
  dispatchable MW into intermittent MW faster than load grows. This
  is the structural stress test the capacity mechanism has to absorb.


In [12]:
from plotly.subplots import make_subplots

fig = make_subplots(
    rows=3, cols=1, shared_xaxes=True,
    vertical_spacing=0.06,
    subplot_titles=(
        "Expected wholesale price ($/MWh)",
        "Fleet missing money ($/yr)",
        "Reserve margin (1.0 = capacity equals peak demand)",
    ),
)
fig.add_trace(go.Scatter(x=df_traj.year, y=df_traj.expected_price, mode="lines", name="price"), 1, 1)
fig.add_trace(
    go.Scatter(x=df_traj.year, y=df_traj.fleet_missing_money_per_year, mode="lines", name="missing money"),
    2, 1,
)
fig.add_trace(
    go.Scatter(x=df_traj.year, y=df_traj.reserve_margin, mode="lines", name="reserve margin"),
    3, 1,
)
fig.add_hline(y=0.15, line_dash="dot", line_color="red", row=3, col=1, annotation_text="NPCC target 0.15")
fig.update_layout(height=720, showlegend=False, title="Ontario 2024-2050 forecast (energy-only, oligopoly)")
fig.show()

### 5.3 Missing money by technology

The fleet aggregate masks a large distributional story. Peakers lose
money almost every year; CCGT flips from winner to loser as gas
prices rise and their dispatch share falls; nuclear gains as SMRs
come online.


In [13]:
per_firm = df_traj.attrs["per_firm"].copy()
per_firm["gap_per_year_musd"] = per_firm["gap_per_year"] / 1e6
pivot_gap = per_firm.pivot_table(index="year", columns="firm", values="gap_per_year_musd", aggfunc="sum")

fig = go.Figure()
for col in pivot_gap.columns:
    fig.add_trace(go.Scatter(x=pivot_gap.index, y=pivot_gap[col], mode="lines", name=col))
fig.add_hline(y=0, line_dash="dot", line_color="black")
fig.update_layout(
    title="Missing money by technology ($M / year). Negative = under-recovering fixed cost.",
    yaxis_title="$M/year",
    xaxis_title="Year",
    height=420,
    legend=dict(orientation="h", y=-0.2),
)
fig.show()

### 5.4 Uncertainty bands (Monte Carlo)

Every pathway anchor past 2030 has genuine uncertainty. We sample
multiplicative lognormal shocks on (peak demand, gas price, renewable
build speed, SMR schedule) and re-run the trajectory. With 200 paths
the P10/P50/P90 spread of missing money widens to roughly **±$2 B/yr
by 2050** — that is the honest policy-relevant uncertainty for a
capacity-remuneration mechanism designer.


In [14]:
mc_df = run_monte_carlo(
    cal,
    years=years,
    config=MonteCarloConfig(n_paths=100, seed=7),
)
bands = summarize_paths(mc_df)


def fan_chart(df, metric, title, ylabel, divisor=1.0):
    fig = go.Figure()
    fig.add_trace(
        go.Scatter(
            x=df.year,
            y=df[f"{metric}_q90"] / divisor,
            mode="lines",
            line=dict(width=0),
            name="P90",
            showlegend=False,
        )
    )
    fig.add_trace(
        go.Scatter(
            x=df.year,
            y=df[f"{metric}_q10"] / divisor,
            mode="lines",
            fill="tonexty",
            line=dict(width=0),
            name="P10-P90",
            fillcolor="rgba(31,119,180,0.25)",
        )
    )
    fig.add_trace(
        go.Scatter(
            x=df.year,
            y=df[f"{metric}_q50"] / divisor,
            mode="lines",
            line=dict(color="#1f77b4", width=2),
            name="P50",
        )
    )
    fig.update_layout(title=title, xaxis_title="Year", yaxis_title=ylabel, height=360)
    return fig


fan_chart(bands, "expected_price", "Expected wholesale price -- fan chart", "$/MWh").show()
fan_chart(
    bands,
    "fleet_missing_money_per_year",
    "Fleet missing money -- fan chart",
    "$B / year",
    divisor=1e9,
).show()

### 5.5 Mechanism ranking over time

Does the same capacity mechanism dominate every year of the pathway,
or does the best policy flip as the fleet mix shifts? Running the
4-mechanism matrix at 2024, 2030, 2040, and 2050 lets us see how
consumer cost and missing money evolve with the structural change.


In [15]:
snap_years = [2024, 2030, 2040, 2050]
snap_traj = [ys for ys in traj if ys.year in snap_years]
mm_mat = run_mechanism_matrix_trajectory(snap_traj, structures=("oligopoly",))
display(
    mm_mat[
        [
            "year",
            "mechanism",
            "expected_price",
            "annual_consumer_cost_for_capacity",
            "fleet_missing_money_per_year",
        ]
    ]
    .round(0)
)

,year,mechanism,expected_price,annual_consumer_cost_for_capacity,fleet_missing_money_per_year
0,2024,Energy-only,82.00,0.00,"4,238,721,098.00"
1,2024,Capacity payment,82.00,"1,893,000,000.00","6,131,721,098.00"
2,2024,Forward capacity,82.00,0.00,"4,238,721,098.00"
3,2024,Reliability options,82.00,"-4,435,837,545.00","-197,116,448.00"
4,2030,Energy-only,67.00,0.00,"3,558,125,315.00"
5,2030,Capacity payment,67.00,"2,119,939,061.00","5,678,064,376.00"
6,2030,Forward capacity,67.00,0.00,"3,558,125,315.00"
7,2030,Reliability options,67.00,"-248,081,213.00","3,310,044,103.00"
8,2040,Energy-only,65.00,0.00,"4,665,976,847.00"
9,2040,Capacity payment,65.00,"2,359,846,154.00","7,025,823,001.00"


### 5.6 Policy takeaways from the forecast

1. **The missing-money problem does not solve itself.** Even in the
   most favourable part of the trajectory (mid-2030s, with new
   nuclear on-line), the fleet-aggregate gap stays in the
   low-billions per year. By 2050 it roughly doubles.

2. **Reserve-margin erosion is the binding constraint.** The default
   pathway respects every publicly announced commitment and still
   walks reserve margin down to ~0.21 by 2050. Either the pathway is
   wrong (we need more firm capacity than announced), or the
   adequacy metric we target must evolve (expected energy unserved
   matters more than nameplate reserve as the fleet intermittent
   share rises).

3. **Mechanism choice is a distributional choice.** Fleet welfare
   under an RO is roughly invariant to strike — what moves is
   *whose* rent gets transferred. Choosing a strike near the Cournot
   expected price minimises total consumer cost while closing the
   peaker gap.

4. **Uncertainty is asymmetric.** P90 missing money climbs almost
   twice as fast as P50 after 2040. Mechanism design should be
   stress-tested against the upper tail, not the central path.
